In [1]:
# ===============================
# 权重网络（即既考虑出现，也考虑出现的次数）
# ===============================

import pandas as pd
import networkx as nx
from scipy.stats import zscore
from pathlib import Path

# ===============================
# 1️⃣ 构建三层网络
# ===============================
def build_multilayer_network(df, zone_col='zone', host_col='host', virus_zol='virus'):
    df = df[[zone_col, host_col, virus_zol]].dropna()

    G = nx.Graph()

    # 节点
    G.add_nodes_from(df[zone_col].unique(), layer='zone')
    G.add_nodes_from(df[host_col].unique(), layer='host')
    G.add_nodes_from(df[virus_zol].unique(), layer='virus')

    # ========== 加权 zone–host ==========
    zh = (
        df.groupby([zone_col, host_col])
        .size()
        .reset_index(name='weight')
    )

    for _, r in zh.iterrows():
        G.add_edge(r[zone_col], r[host_col], weight=r['weight'])

    # ========== 加权 host–virus ==========
    hv = (
        df.groupby([host_col, virus_zol])
        .size()
        .reset_index(name='weight')
    )

    for _, r in hv.iterrows():
        G.add_edge(r[host_col], r[virus_zol], weight=r['weight'])

    return G

# ===============================
# 2️⃣ 计算宿主跨层重要性 HII
# ===============================
def compute_host_HII(G, hosts, export_excel=None):

    # 加权 zone 强度
    host_zone_strength = {
        h: sum(
            G[h][n].get('weight', 1)
            for n in G.neighbors(h)
            if G.nodes[n]['layer'] == 'zone'
        )
        for h in hosts
    }

    # 加权 virus 强度
    host_virus_strength = {
        h: sum(
            G[h][n].get('weight', 1)
            for n in G.neighbors(h)
            if G.nodes[n]['layer'] == 'virus'
        )
        for h in hosts
    }

    # betweenness（保持无权重，更稳健）
    betweenness = nx.betweenness_centrality(G, normalized=True)
    host_betweenness = {h: betweenness.get(h, 0) for h in hosts}

    host_metrics = pd.DataFrame({
        'zone_strength': host_zone_strength,
        'virus_strength': host_virus_strength,
        'betweenness': host_betweenness
    }).fillna(0)

    host_metrics['HII'] = (
        zscore(host_metrics['zone_strength']) +
        zscore(host_metrics['virus_strength']) +
        zscore(host_metrics['betweenness'])
    )

    host_metrics = (
        host_metrics
        .sort_values('HII', ascending=False)
        .reset_index()
        .rename(columns={'index': 'host'})
    )

    if export_excel:
        output_dir = Path('output/2_multilayer_centrality')
        output_dir.mkdir(parents=True, exist_ok=True)
        host_metrics.to_excel(output_dir / export_excel, index=False)

    return host_metrics

# ===============================
# 3️⃣ 计算病原体跨层扩散 PSI
# ===============================
def compute_virus_PSI(G, viruss, hosts, export_excel=None):

    # 加权宿主强度
    virus_host_strength = {
        v: sum(
            G[v][n].get('weight', 1)
            for n in G.neighbors(v)
            if G.nodes[n]['layer'] == 'host'
        )
        for v in viruss
    }

    # 空间覆盖（仍用 presence-based）
    virus_zone_coverage = {}
    for v in viruss:
        zone_set = set()
        for h in G.neighbors(v):
            if G.nodes[h]['layer'] == 'host':
                zone_set.update(
                    n for n in G.neighbors(h)
                    if G.nodes[n]['layer'] == 'zone'
                )
        virus_zone_coverage[v] = len(zone_set)

    # 加权特征向量中心性
    eigenvector = nx.eigenvector_centrality_numpy(G, weight='weight')
    virus_eigen = {v: eigenvector.get(v, 0) for v in viruss}

    virus_metrics = pd.DataFrame({
        'host_strength': virus_host_strength,
        'n_zones': virus_zone_coverage,
        'eigenvector': virus_eigen
    }).fillna(0)

    virus_metrics['PSI'] = (
        zscore(virus_metrics['host_strength']) +
        zscore(virus_metrics['n_zones']) +
        zscore(virus_metrics['eigenvector'])
    )

    virus_metrics = (
        virus_metrics
        .sort_values('PSI', ascending=False)
        .reset_index()
        .rename(columns={'index': 'virus'})
    )

    if export_excel:
        output_dir = Path('output/2_multilayer_centrality')
        output_dir.mkdir(parents=True, exist_ok=True)
        virus_metrics.to_excel(output_dir / export_excel, index=False)

    return virus_metrics


# ===============================
# 4️⃣ 封装接口
# ===============================
def compute_network_metrics(df, zone_col='zone', host_col='host', virus_zol='virus',
                            host_excel=None, virus_excel=None):

    G = build_multilayer_network(df, zone_col, host_col, virus_zol)
    hosts = df[host_col].dropna().unique()
    viruss = df[virus_zol].dropna().unique()

    host_metrics = compute_host_HII(G, hosts, export_excel=host_excel)
    virus_metrics = compute_virus_PSI(G, viruss, hosts, export_excel=virus_excel)

    return host_metrics, virus_metrics, G


# ===============================
# 使用示例
# ===============================
if __name__ == "__main__":
    df = pd.read_excel(r'input/host_virus-NEW-0313.xlsx')
    df = df[df['virus_new'].notna()].reset_index(drop=True)
    df = df[['zone', 'Host_species', 'virus_new', 'County']]
    df = df.rename(columns={'Host_species': 'host', 'virus_new': 'virus'})
    df_new = (
        df
        .assign(virus=df['virus'].str.split(','))
        .explode('virus')
        .assign(virus=lambda x: x['virus'].str.strip())
        .reset_index(drop=True)
    )
    df_new[['host', 'virus']] = df_new[['host', 'virus']].apply(lambda x: x.str.strip())
    # df_new=df_new.drop_duplicates().reset_index(drop=True)

    host_metrics, virus_metrics, G = compute_network_metrics(df_new, host_excel='host_centrality_weighted.xlsx', virus_excel='virus_centrality_weighted.xlsx')